# Movie Revenue Prediction — Phase 2
### Machine Learning on TMDB Dataset

**Goal:** Build a model that predicts how much revenue a movie will earn, given its budget, genre, runtime, and release year.

**Models we'll compare:**
- Linear Regression — simple baseline
- Random Forest — powerful ensemble model

---

## 1. Imports

**What & Why:**
- `pandas` / `numpy` — data handling (same as Phase 1)
- `matplotlib` / `seaborn` — visualizations
- `sklearn` (scikit-learn) — the most popular ML library in Python. Contains ready-made implementations of every major ML algorithm so you don't build them from scratch.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from matplotlib.patches import Patch

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

print('All imports successful')

## 2. Load & Clean Data

**What & Why:**
We reuse the same cleaning steps from Phase 1 EDA. This is standard in real projects — the cleaning logic gets reused every time you build a new model on the same dataset.

New step: we drop rows where budget or revenue is missing.
Why? ML models cannot handle missing values — they need complete rows to learn from.

In [ ]:
df = pd.read_csv('tmdb_5000_movies.csv')

df['budget'] = df['budget'].replace(0, np.nan)
df['revenue'] = df['revenue'].replace(0, np.nan)
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year

def extract_names(json_str):
    try:
        items = ast.literal_eval(json_str)
        return [item['name'] for item in items]
    except:
        return []

df['genre_list'] = df['genres'].apply(extract_names)
df['primary_genre'] = df['genre_list'].apply(lambda x: x[0] if x else 'Unknown')

df_ml = df.dropna(subset=['budget', 'revenue', 'runtime', 'release_year'])

print(f'Rows available for ML: {len(df_ml):,} (out of {len(df):,} total)')
df_ml[['title', 'budget', 'revenue', 'runtime', 'release_year', 'primary_genre']].head()

## 3. Feature Engineering

**What is Feature Engineering?**
Creating new input variables from raw data to help the model learn better.

**log_budget / log_revenue:** Budgets range from $1,000 to $300M — a massive range. Log compresses this so the model is not overwhelmed by extreme values. Think of it like a map scale.

**One-hot encoding:** Converts text genres to numbers. 'Action' becomes a column with 1 or 0.

We predict log(revenue) and convert back to dollars at the end.

In [ ]:
df_ml = df_ml.copy()
df_ml['log_budget'] = np.log1p(df_ml['budget'])
df_ml['log_revenue'] = np.log1p(df_ml['revenue'])

genre_dummies = pd.get_dummies(df_ml['primary_genre'], prefix='genre')
print(f'Genres found: {list(genre_dummies.columns)}')

features = pd.concat([
    df_ml[['log_budget', 'runtime', 'release_year', 'vote_count']],
    genre_dummies
], axis=1)

target = df_ml['log_revenue']

print(f'\nFeature matrix shape: {features.shape[0]} movies x {features.shape[1]} features')
features.head(3)

## 4. Train / Test Split

**What & Why:**
Think of it like an exam. You study from a textbook (training data), then get tested on questions you have never seen (test data).

- Training set (80%) — model learns patterns from this
- Test set (20%) — hidden from model, used to check if predictions are accurate

If we tested on training data, the model would memorize answers and score perfectly — but fail on real new movies. That is called overfitting.

random_state=42 fixes the random split so results are reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=42
)

print(f'Training set: {X_train.shape[0]:,} movies  <- model learns from these')
print(f'Test set:     {X_test.shape[0]:,} movies  <- model is evaluated on these')

## 5. Model 1 — Linear Regression

**What it is:**
The simplest ML model. It finds the best straight line through your data.

revenue = (w1 x budget) + (w2 x runtime) + (w3 x year) + ...

Each w (weight/coefficient) tells you how much that feature contributes. The model finds best weights by minimizing prediction errors.

**Why start here:** It is fast, interpretable, and gives us a baseline — a minimum score any better model must beat.

**R2 score explained:**
- 1.0 = perfect predictions
- 0.0 = no better than just guessing the average
- 0.6 = explains 60% of why revenues differ between movies

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)   # .fit() = learn from training data

lr_predictions = lr_model.predict(X_test)   # .predict() = guess on unseen data

lr_r2 = r2_score(y_test, lr_predictions)

actual_revenue = np.expm1(y_test)
predicted_revenue_lr = np.expm1(lr_predictions)
lr_rmse_dollars = np.sqrt(mean_squared_error(actual_revenue, predicted_revenue_lr))

print('Linear Regression Results:')
print(f'  R2 Score: {lr_r2:.3f}  <- explains {lr_r2*100:.1f}% of revenue variance')
print(f'  RMSE: ${lr_rmse_dollars/1e6:.1f}M  <- average prediction error in dollars')

## 6. Model 2 — Random Forest

**What it is:**
Builds 100 decision trees and averages their predictions.

A single decision tree asks yes/no questions:
- Is budget > $50M? Yes -> Is it Action? Yes -> Predict $150M

One tree overfits. But 100 trees each trained on a slightly different random sample, then averaged, is very robust. Like asking 100 experts instead of one.

This idea of combining many weak models into one strong model is called ensemble learning — one of the most powerful ideas in ML.

n_estimators=100 means 100 trees. More trees = more accurate but slower.

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1   # use all CPU cores to speed up training
)
rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)
rf_r2 = r2_score(y_test, rf_predictions)
predicted_revenue_rf = np.expm1(rf_predictions)
rf_rmse_dollars = np.sqrt(mean_squared_error(actual_revenue, predicted_revenue_rf))

print('Random Forest Results:')
print(f'  R2 Score: {rf_r2:.3f}  <- explains {rf_r2*100:.1f}% of revenue variance')
print(f'  RMSE: ${rf_rmse_dollars/1e6:.1f}M  <- average prediction error in dollars')
print(f'\nImprovement over Linear Regression: +{(rf_r2 - lr_r2)*100:.1f}% R2')

## 7. Visualize Results

**Actual vs Predicted plot:**
- X axis = real revenue (what actually happened)
- Y axis = what our model predicted

A perfect model would have all points on the diagonal red line (predicted = actual).
The more scattered the points, the worse the model.
Points above the line = model over-predicted. Below = under-predicted.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Model comparison
models = ['Linear Regression', 'Random Forest']
r2_scores = [lr_r2, rf_r2]
colors = ['steelblue', 'mediumseagreen']
bars = axes[0].bar(models, r2_scores, color=colors, width=0.4, edgecolor='white')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('R2 Score (higher = better)')
axes[0].set_title('Model comparison')
for bar, score in zip(bars, r2_scores):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{score:.3f}', ha='center', fontsize=11, fontweight='500')

# Plot 2: Linear Regression actual vs predicted
axes[1].scatter(np.log10(actual_revenue + 1), np.log10(predicted_revenue_lr + 1),
                alpha=0.3, s=10, color='steelblue')
max_val = max(np.log10(actual_revenue + 1).max(), np.log10(predicted_revenue_lr + 1).max())
axes[1].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
axes[1].set_xlabel('Actual revenue (log)')
axes[1].set_ylabel('Predicted revenue (log)')
axes[1].set_title(f'Linear Regression (R2={lr_r2:.2f})')
axes[1].legend(fontsize=9)

# Plot 3: Random Forest actual vs predicted
axes[2].scatter(np.log10(actual_revenue + 1), np.log10(predicted_revenue_rf + 1),
                alpha=0.3, s=10, color='mediumseagreen')
axes[2].plot([0, max_val], [0, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
axes[2].set_xlabel('Actual revenue (log)')
axes[2].set_ylabel('Predicted revenue (log)')
axes[2].set_title(f'Random Forest (R2={rf_r2:.2f})')
axes[2].legend(fontsize=9)

plt.tight_layout()
plt.savefig('plot_model_comparison.png', bbox_inches='tight')
plt.show()

## 8. Feature Importance

**What it is:**
After training a Random Forest, you can ask: which features did you rely on most?

This is incredibly useful because:
- It reveals what actually drives the outcome
- Helps you remove useless features to simplify the model
- Gives business insight: is it budget or genre that matters more?

All importance scores sum to 1.0 across all features.

In [ ]:
importance_df = pd.DataFrame({
    'feature': features.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(15)

colors = ['mediumseagreen' if not x.startswith('genre') else 'mediumpurple'
          for x in importance_df['feature']]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'], importance_df['importance'],
        color=colors, edgecolor='white')
ax.invert_yaxis()
ax.set_xlabel('Importance score')
ax.set_title('Feature importance — what drives revenue predictions?\n(green = numeric, purple = genre)')

legend_elements = [Patch(facecolor='mediumseagreen', label='Numeric feature'),
                   Patch(facecolor='mediumpurple', label='Genre feature')]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig('plot_feature_importance.png', bbox_inches='tight')
plt.show()

print('Top 5 most important features:')
for _, row in importance_df.head(5).iterrows():
    print(f'  {row["feature"]:<25} {row["importance"]:.3f}')

## 9. Make a Real Prediction

**The fun part** — use the trained model to predict revenue for a made-up movie.

This is exactly what a production AI system does: a user inputs movie details, the model returns a predicted revenue. Studios use similar models to decide whether to greenlight a film.

In [ ]:
new_movie = {
    'title': 'My Hypothetical Blockbuster',
    'budget': 150_000_000,
    'runtime': 120,
    'release_year': 2025,
    'vote_count': 0,
    'primary_genre': 'Action'
}

new_row = pd.DataFrame(columns=features.columns).reindex([0]).fillna(0)
new_row['log_budget'] = np.log1p(new_movie['budget'])
new_row['runtime'] = new_movie['runtime']
new_row['release_year'] = new_movie['release_year']
new_row['vote_count'] = new_movie['vote_count']

genre_col = f"genre_{new_movie['primary_genre']}"
if genre_col in new_row.columns:
    new_row[genre_col] = 1

log_pred = rf_model.predict(new_row)[0]
predicted_revenue = np.expm1(log_pred)
roi = ((predicted_revenue - new_movie['budget']) / new_movie['budget']) * 100

print(f"Movie:             {new_movie['title']}")
print(f"Genre:             {new_movie['primary_genre']}")
print(f"Budget:            ${new_movie['budget']/1e6:.0f}M")
print(f"Predicted Revenue: ${predicted_revenue/1e6:.1f}M")
print(f"Predicted ROI:     {roi:.1f}%")
print(f"Verdict:           {'Profitable' if roi > 0 else 'Loss'}")

## 10. Summary

**What you built:**
A complete ML pipeline: raw data -> features -> train/test split -> two models -> evaluation -> prediction

**What you learned:**
- EDA (Phase 1) feeds directly into ML — cleaning done once, reused
- Feature engineering (log transform, one-hot encoding) improves model performance
- Random Forest outperforms Linear Regression on complex real-world data
- Feature importance reveals what actually drives predictions

**Fill in your results:**
- R2 of Linear Regression: ...
- R2 of Random Forest: ...
- Most important feature: ...
- Predicted revenue for hypothetical movie: ...